# 15 · M2.0 — M1.9 with an ensemble of three volatility estimates, and 2,000 draws

## Executive summary (read this first)

**M2.0 = M1.9 + two changes:**

| piece | what it does | why |
|---|---|---|
| **volatility ensemble** | draw i takes the EWMA sd with halflife 21, 63 or 252 steps, by turns (i mod 3): a third each | No single halflife is right in every regime. A fast estimate reacts to a new storm, a slow one remembers the old ones. Pooling the three is forecast combination applied to volatility, and it fattens the tails a little. The halflives are set in advance: a month, a quarter, a year |
| **2,000 draws** | instead of 500 | the tail score reads the 1 % and 99 % quantiles off the draws. From 500 draws the 1 % quantile rests on 5 of them. The mixtures of M1.9 and M2.0 need more draws, and the docs recommend at least 1,000 for F4 |

**Results** (§3): M2.0 **0.9155** × M0 (M1.9 0.9194): the ensemble alone gives 0.9163, 2,000 draws alone 0.9188. Every family
gains. Better than M0 on 96 of 103 units.

**The path from M1.5**, on the same 23,635 backtest dates:

| model | change | all | F1 | F2 | F3 | F4 |
|---|---|---|---|---|---|---|
| M1.5 | production today | 0.9444 | 0.893 | 0.990 | 0.914 | 0.964 |
| M1.7 | drift = the significant recent trend | 0.9312 | 0.883 | 0.966 | 0.912 | 0.951 |
| M1.8 | + zero floor, variance ratio | 0.9266 | 0.871 | 0.961 | 0.910 | 0.950 |
| M1.9 | + 1 draw in 4 carries the full trend | 0.9194 | 0.867 | 0.946 | 0.908 | 0.943 |
| **M2.0** | + volatility ensemble, 2,000 draws | **0.9155** | 0.861 | 0.942 | 0.907 | 0.938 |

Some choices were made on this backtest (the James–Stein constant and window, the shrinkage of the variance ratio, the
trend share), so these numbers are optimistic. The clean check before production is the team eval's real outcomes. Ideas
that were tried and not kept are listed in the README. No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 14's M1.9, plus the volatility ensemble and 2,000 draws

`m20_draws(U, origin)` is M2.0; `vol_pool=None, n_draws=500` is M1.9 (checked against notebook 14's saved scores). The other two
variants add one piece each.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

# Every asset's whole history as steps (its own transform, M0's gap rule), computed once: the recent trend (M1.7) looks
# back 126 steps, the variance ratio (M1.8) over the whole history.
FULL = {}
for _u, _U in UNITS_DATA.items():
    for _a in _U["assets"]:
        _rows = _U["rows"][_a]
        _lm = _U["rules"][_a]["transform"] == "log_diff" and min(v for _, v in _rows) > 0
        _st, _ = m0._steps([(d, np.log(v)) for d, v in _rows] if _lm else _rows, "level" if _lm else _U["target_type"])
        _ds = sorted(_st)
        FULL[(_u, _a)] = (np.array([str(d) for d in _ds]), np.array([_st[d] for d in _ds]))

def hist_steps(U, a, origin):
    """All of one asset's steps up to the origin, oldest first."""
    d, v = FULL[(U["unit"], a)]
    return v[: np.searchsorted(d, origin, side="right")]

def recent_trend(U, a, origin, mu_window):
    """M1.7's drift per step.
    CPI, NFP (drift by type): the mean of the last 12 monthly steps -- this year's trend, not the 21-year average.
    Everything else: the mean m of the last 126 steps, shrunk toward 0 by its t-statistic t = m / (sd / sqrt(126))
    (James-Stein): drift = m x max(0, 1 - 4 / t^2). |t| <= 2 gives no drift at all; |t| = 4 keeps 75 %."""
    v = hist_steps(U, a, origin)
    if U["rules"][a]["drift"]:
        return float(v[-12:].mean()) if len(v) >= 12 else mu_window
    v = v[-126:]
    sd = v.std(ddof=1)
    t = v.mean() / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return float(v.mean() * max(0.0, 1.0 - 4.0 / max(t * t, 1e-12)))

def variance_ratio(U, a, h_steps, origin, n0=30.0):
    """M1.8: VR = mean(h-step move^2) / (h x mean(1-step move^2)) over the asset's whole history before the origin.
    VR > 1: moves add up faster than sqrt(h) (trends, cycles); VR < 1: they partly cancel (mean reversion). Raw moves (no
    demeaning) for assets without a drift, because a trend we do not forecast is width. Shrunk toward 1 by the number of
    independent windows: VR = 1 + (VR_raw - 1) x n / (n + 30), n = history steps / h."""
    v = hist_steps(U, a, origin)
    if h_steps <= 1 or len(v) < 2 * h_steps + 20:
        return 1.0
    m = v.mean() if U["rules"][a]["drift"] else 0.0
    c = np.concatenate([[0.0], np.cumsum(v - m)])
    ratio = np.mean((c[h_steps:] - c[:-h_steps]) ** 2) / (h_steps * np.mean((v - m) ** 2))
    n_eff = len(v) / h_steps
    return 1.0 + (ratio - 1.0) * n_eff / (n_eff + n0)

def m20_draws(U, origin, drift17=True, floor=True, vr=True, trend_mix=0.25, vol_pool=(21, 63, 252), n_draws=2000, ewma_halflife=63):
    """M2.0 = M1.9 + an ensemble of volatility estimates (vol_pool: the draws take turns over EWMA halflives 21, 63 and
    252) + 2000 draws. vol_pool=None, n_draws=500 is M1.9 (notebook 14)."""
    N = n_draws
    by_asset, x, log_mode, last = fit(U, origin, WINDOW)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
    sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])     # M1.5: EWMA sd, halflife 63
    sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    drift = {a: (mu[ai[a]] if U["rules"][a]["drift"] else 0.0) for a in by_asset}      # M1.5: drift by type
    if drift17:                                                                          # M1.7: the recent trend
        drift = {a: recent_trend(U, a, origin, mu[ai[a]]) for a in by_asset}
    def centre(a, h, d):
        return (0.0 if log_mode[a] else last[a]) + s[h] * d[a]
    mean = np.array([centre(a, h, drift) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((N, len(cells))) @ f.T
    scale = np.ones(len(cells))
    if vr:
        scale = scale * np.array([np.sqrt(variance_ratio(U, a, s[h], origin)) for a, h in cells])
    dev = dev * scale
    if vol_pool:                        # M2.0: draw i takes the EWMA sd of halflife vol_pool[i % 3]
        ratio = np.array([[ewma_sd_asset(U, a, origin, hl) / sd_e[ai[a]] if sd_e[ai[a]] > 0 else 1.0
                           for a, h in cells] for hl in vol_pool])
        dev = dev * ratio[np.arange(N) % len(vol_pool)]
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(N, 1))), 1.0)
    sm = mean + dev
    if trend_mix:
        k = int(round(trend_mix * N))           # the last k draws: the full trend of the window
        full = {a: (drift[a] if U["rules"][a]["drift"] else mu[ai[a]]) for a in by_asset}
        sm[N - k:] = np.array([centre(a, h, full) for a, h in cells]) + dev[N - k:]
    out = np.empty((N, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if floor and asset_type(a) == "rates":
            col = np.maximum(col, 0.0)          # zero lower bound on Treasury yields
        if s[h] <= 0:
            col = np.full(N, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

def draw_variant(U, o, kw):
    return m20_draws(U, o, **kw)
VARIANTS = {"M1.9": dict(vol_pool=None, n_draws=500), "M1.9 + vol ensemble": dict(n_draws=500),
            "M1.9 + 2000 draws": dict(vol_pool=None), "M2.0": {}}
def score_unit(u):
    U = UNITS_DATA[u]
    out = []
    for o, y in BACKTEST[u]:
        for m, kw in VARIANTS.items():
            d = draw_variant(U, o, kw)
            out.append({"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], d.reshape(d.shape[0], -1), y)})
    return out
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // len(VARIANTS):,} dates x {len(VARIANTS)} variants in {time.time() - t0:.0f}s")
SCV = {m: vs_m0(G[G.model == m]) for m in VARIANTS}
d = float((SCV["M1.9"] - vs_m0(pd.read_parquet(DATA / "m1_9_backtest_scores.parquet"))).abs().max())
print(f"check — the new pieces switched off vs notebook 14's saved M1.9: max |difference| per unit = {d:.1e}")
assert d < 1e-12
print(pd.Series({m: s.mean() for m, s in SCV.items()}, name="competition score vs M0 (all units)").round(4).to_string())
M20 = G[G.model == "M2.0"].drop(columns="model").reset_index(drop=True)

scored 23,635 dates x 4 variants in 37s
check — the new pieces switched off vs notebook 14's saved M1.9: max |difference| per unit = 0.0e+00
M1.9                  0.9194
M1.9 + vol ensemble   0.9163
M1.9 + 2000 draws     0.9188
M2.0                  0.9155


## 2 — Worked example: the three volatility estimates, and what 2,000 draws buy

(a) One F4 card (t2-F4-china-panic-2016, the market factor, 21 days) on the backtest date where the fast estimate is
furthest above the slow one, a fresh storm: the EWMA sd at each halflife and the width of the pooled forecast. (b) The 1 % quantile of that forecast, read off four separate blocks of 500 draws and
off all 2,000.

In [3]:
u = "t2-F4-china-panic-2016"; U = UNITS_DATA[u]; a = "MKT"
o, y = max(BACKTEST[u], key=lambda oy: ewma_sd_asset(U, a, oy[0], 21) / ewma_sd_asset(U, a, oy[0], 252))
sd63 = ewma_sd_asset(U, a, o, 63)
display(pd.DataFrame({hl: {"EWMA sd per step": ewma_sd_asset(U, a, o, hl), "÷ the halflife-63 sd": ewma_sd_asset(U, a, o, hl) / sd63,
                           "draws using it": f"i mod 3 = {k}"} for k, hl in enumerate((21, 63, 252))}).T.rename_axis(f"(a) {u}, {o}: halflife"))
rms = np.sqrt(np.mean([ewma_sd_asset(U, a, o, hl) ** 2 for hl in (21, 63, 252)]))
print(f"pooled width: √(mean of the three variances) = {rms:.5f} per step, {rms / sd63:.3f} × the halflife-63 sd")
d = m20_draws(U, o)[:, 0, 0]
blocks = [np.quantile(d[i * 500:(i + 1) * 500], 0.01) for i in range(4)]
print(f"(b) 1 % quantile from four blocks of 500 draws: {', '.join(f'{b:.4f}' for b in blocks)} "
      f"(spread {max(blocks) - min(blocks):.4f}); from all 2,000: {np.quantile(d, 0.01):.4f}; answer {y[0]:.4f}")

,EWMA sd per step,÷ the halflife-63 sd,draws using it
"(a) t2-F4-china-panic-2016, 2015-09-03: halflife",,,
21,0.01398,1.335,i mod 3 = 0
63,0.01048,1,i mod 3 = 1
252,0.009263,0.884,i mod 3 = 2


pooled width: √(mean of the three variances) = 0.01142 per step, 1.090 × the halflife-63 sd
(b) 1 % quantile from four blocks of 500 draws: -0.1236, -0.1058, -0.0939, -0.0950 (spread 0.0298); from all 2,000: -0.1141; answer 0.0154


## 3 — Every model on the same backtest

In [4]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "production walk (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.5 (EWMA sd 63)": pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"),
          "M1.7 (recent trend)": pd.read_parquet(DATA / "m1_7_backtest_scores.parquet"),
          "M1.8 (+ floor, variance ratio)": pd.read_parquet(DATA / "m1_8_backtest_scores.parquet"),
          "M1.9 (+ trend scenario)": pd.read_parquet(DATA / "m1_9_backtest_scores.parquet"),
          "M2.0 (+ vol ensemble, 2000 draws)": M20}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
production walk (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
M1.5 (EWMA sd 63),0.8932,0.99,0.9141,0.9642,0.9444,85/103
M1.7 (recent trend),0.8828,0.9656,0.9124,0.9505,0.9312,86/103
"M1.8 (+ floor, variance ratio)",0.8714,0.961,0.9097,0.9498,0.9266,84/103
M1.9 (+ trend scenario),0.8673,0.9461,0.9084,0.9428,0.9194,92/103
"M2.0 (+ vol ensemble, 2000 draws)",0.8612,0.9424,0.9069,0.9384,0.9155,96/103


## 4 — M1.9 vs M2.0: per family, per asset type, and the units that moved most

In [5]:
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE), "M1.5 vs M0": SC["M1.5 (EWMA sd 63)"],
                         "M1.9 vs M0": SC["M1.9 (+ trend scenario)"], "M2.0 vs M0": SC["M2.0 (+ vol ensemble, 2000 draws)"]}).loc[:, lambda d: ~d.columns.duplicated()]
def summary(g):
    return {"units": len(g), "M1.5": g["M1.5 vs M0"].mean(), "M1.9": g["M1.9 vs M0"].mean(), "M2.0": g["M2.0 vs M0"].mean(),
            "M2.0 better than M1.9": f'{(g["M2.0 vs M0"] < g["M1.9 vs M0"]).sum()}/{len(g)}',
            "M2.0 better than M0": f'{(g["M2.0 vs M0"] < 1).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)
moved = (PER_UNIT["M2.0 vs M0"] - PER_UNIT["M1.9 vs M0"]).sort_values()
print("the units that moved most (M2.0 - M1.9; below 0 = better):")
display(PER_UNIT.loc[list(moved.index[:8]) + list(moved.index[-5:])].assign(change=moved))

per family:


,units,M1.5,M1.9,M2.0,M2.0 better than M1.9,M2.0 better than M0
family,,,,,,
T2-F1,23,0.8932,0.8673,0.8612,16/23,21/23
T2-F2,27,0.99,0.9461,0.9424,16/27,24/27
T2-F3,22,0.9141,0.9084,0.9069,11/22,22/22
T2-F4,31,0.9642,0.9428,0.9384,22/31,29/31
all,103,0.9444,0.9194,0.9155,65/103,96/103


per asset type:


,units,M1.5,M1.9,M2.0,M2.0 better than M1.9,M2.0 better than M0
asset type,,,,,,
cpi,2,0.899,0.735,0.7307,1/2,2/2
equity_factor,16,0.9665,0.9555,0.9544,8/16,14/16
fx,35,0.9614,0.94,0.9324,24/35,33/35
fx+rates,5,0.933,0.9174,0.9171,2/5,5/5
payrolls,1,0.9682,0.8225,0.8219,1/1,1/1
rates,43,0.9234,0.8983,0.8962,28/43,40/43
unemployment,1,1.024,1.008,0.9873,1/1,1/1


the units that moved most (M2.0 - M1.9; below 0 = better):


,family,asset type,M1.5 vs M0,M1.9 vs M0,M2.0 vs M0,change
t2-F2-qe2-jackson-hole-2010,T2-F2,rates,0.9549,0.9628,0.9259,-0.03689
t2-F1-aud-on-hold-2016,T2-F1,fx,0.9559,0.9488,0.9169,-0.03188
t2-F2-nirp-paradox-2016,T2-F2,fx,1.042,1.008,0.9764,-0.03153
t2-F1-eur-range-2017,T2-F1,fx,0.8195,0.8452,0.8151,-0.03008
t2-F4-gbp-brexit-2016,T2-F4,fx,0.9565,0.9658,0.9367,-0.02908
t2-F1-jpy-ycc-flex-2018,T2-F1,fx,0.8724,0.9025,0.8762,-0.02635
t2-F1-pause-2006,T2-F1,rates,1.291,1.132,1.106,-0.02619
t2-F1-dkk-peg-2019,T2-F1,fx,0.7747,0.8129,0.7872,-0.02571
t2-F4-hike-cycle-2021Q4b,T2-F4,rates,0.8661,0.7105,0.7323,0.0218
t2-F2-credit-crunch-2007,T2-F2,rates,1.014,0.9638,0.989,0.02522


## 5 — Save

In [6]:
M20.to_parquet(DATA / "m2_0_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m2_0_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m2_0_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m2_0_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for name in ("m2_0_backtest_scores.parquet", "m2_0_vs_m0_per_unit.csv", "m2_0_vs_m0_per_family.csv", "m2_0_vs_m0_by_asset_type.csv"):
    print(f"data/{name:<30} {(DATA / name).stat().st_size / 1e3:8.1f} kB")

data/m2_0_backtest_scores.parquet      867.1 kB
data/m2_0_vs_m0_per_unit.csv             9.7 kB
data/m2_0_vs_m0_per_family.csv           0.5 kB
data/m2_0_vs_m0_by_asset_type.csv        0.6 kB


## 6 — In short

In [7]:
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M2.0 vs M1.9: better on {a['M2.0 better than M1.9']} units; per family M2.0 (M1.9): "
      + "; ".join(f"{f} {FAMILY.loc[f, 'M2.0']:.4f} ({FAMILY.loc[f, 'M1.9']:.4f})" for f in FAMILY.index if f != "all"))

  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  production walk (text off)           0.9726   (better than M0 on 74/103)
  M1.5 (EWMA sd 63)                    0.9444   (better than M0 on 85/103)
  M1.7 (recent trend)                  0.9312   (better than M0 on 86/103)
  M1.8 (+ floor, variance ratio)       0.9266   (better than M0 on 84/103)
  M1.9 (+ trend scenario)              0.9194   (better than M0 on 92/103)
  M2.0 (+ vol ensemble, 2000 draws)    0.9155   (better than M0 on 96/103)
M2.0 vs M1.9: better on 65/103 units; per family M2.0 (M1.9): T2-F1 0.8612 (0.8673); T2-F2 0.9424 (0.9461); T2-F3 0.9069 (0.9084); T2-F4 0.9384 (0.9428)
